# Table B

Table B represents the cleaned collection data to analyze the bin fill data in Table A to observe high and low-participation regions and plan campaigns based on their recycling habits.

## Data Source
The dataset used in this project is the ultrasonic waste bin sensor dataset published on Zenodo:
https://zenodo.org/records/14988663

The dataset contains Boolean flags and cycle numbers for the collection timestamps, fill percentage, and collection statistical measures, including the stability and consistency of the fill-level signal between two collection times (Avg_Dist) and the Spearman's rank coefficient correlation.

For this project, we use both the cleaned bin fill data from Table A and the corrected collection metrics provided by the dataset authors. This includes feature engineering to create additional features to be used for future predictions.

## Table B Schema

Table B contains the following columns:

- ContainerID
- cycle_duration_days
- Avg_Dist
- Spearman
- collection_fill_percentage
- avg_daily_fill_growth
- overflow_flag

## Field Definitions
| Field | Description |
|------|-------------|
| ContainerID | Unique identifier for each waste bin.|
| cycle_duration_days | Number of days in a fill cycle. |
| AVG_DIST | Summary statistic measure of the stability and consistency of the fill-level signal between two collections. A higher AVG_DIST value signals more trustworthy data. |
| Spearman | Spearman's rank correlation coefficient. This value is calculated using fill level values and the amount of time between collections. |
| collection_fill_percentage | Percentage indicating how full the bin is prior to collection.|
| avg_daily_fill_growth | How fast the bin fills daily. |
| overflow_flag | Boolean flag to indicate if fill level is over or equal 80%. |

In [1]:
# Imports & Setup
from datasets import load_dataset
from huggingface_hub import list_repo_files
import pandas as pd
import os
import logging
from datasets.utils.logging import disable_progress_bar

# to suppress Hugging Face info messages due to missing yaml metadata
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
disable_progress_bar()

In [2]:
# Dataset Sources
repo_id = "SA61team5/ultrasonic-waste-bin-sensor-raw"
fill_df = pd.read_csv("Cleaned-data/tableA.csv")
selected_container_ids = pd.read_csv("Cleaned-data/selected_container_ids.csv")["ContainerID"].astype(str).tolist()

all_files = list_repo_files(repo_id, repo_type="dataset")

# Select corrected recs csv files only, matching selected container IDs from Table A
recs_files = [f for f in all_files 
              if "_recs_Corrected_with_metrics" in f 
              and f.endswith(".csv")
              and os.path.basename(f).split("_")[1] in selected_container_ids]

# Limit to 30 bins for this prototype
recs_files = recs_files[:30]

## Step 1: Per-File Loading and Cleaning

For each selected bin file, we:
- Load the corrected recs data
- Extract the container identifier from the filename
- Calculate the number of days between each cycle

We then combine the files into one dataframe (combined_recs_df).

In [3]:
all_recs_dfs = []

for file_path in recs_files:
    dataset = load_dataset(repo_id, data_files=file_path)
    df = pd.DataFrame(dataset["train"][:])

    container_id = os.path.basename(file_path).split("_")[1]
    df["ContainerID"] = container_id
    df["ContainerID"] = df["ContainerID"].astype("int64")

    df["timestamp"] = pd.to_datetime(df["Date"])
    
    df["cycle_duration_days"] = (df['timestamp'].diff().dt.total_seconds() / (24 * 3600)).fillna(0).round().astype(int)

    df.drop(columns=['Date'], axis=1, inplace=True)

    cols = ['ContainerID', 'timestamp', 'End_Pointer', 'cycle_duration_days', 'Avg_Dist', 'Spearman']

    df = df[cols]
    
    all_recs_dfs.append(df)

combined_recs_df = pd.concat(all_recs_dfs, ignore_index=True)

## Step 2: Filtering Rows and Getting Data from Table A

With Table A, we:
- Create a dataframe (fill_collect_df) to keep rows where collection was done
- Create a dataframe (pre_fill_collect_df) to keep rows *before* collection was done
- Used these two dataframes with the dataframe in Step 1 for cleaning

In [4]:
#make df when Rec == 1 (collection rows)
fill_collect_df = fill_df[fill_df['Rec'] == 1].copy()
fill_collect_df = fill_collect_df.drop(fill_collect_df.index[0])

#make df before collection (get fill_percentage before collection)
fill_indices = fill_collect_df.index
pre_fill_collect_df = fill_df.loc[fill_indices[fill_indices!= 0] - 1].copy()

#filter only rows in combined_recs_df that are in fill_collect_df, using indexes
combined_recs_df["Index"] = combined_recs_df["End_Pointer"]
filtered_recs_df = combined_recs_df.merge(
    fill_df[["ContainerID", "Index"]],
    on=["ContainerID", "Index"],
    how="inner"
)
filtered_recs_df = filtered_recs_df.drop(filtered_recs_df.index[0])

filtered_recs_df = filtered_recs_df.drop(columns=["Index", "End_Pointer", "timestamp"])

filtered_recs_df["collection_fill_percentage"] = pre_fill_collect_df["fill_percentage"].values
filtered_recs_df['avg_daily_fill_growth'] = (filtered_recs_df['collection_fill_percentage'] / filtered_recs_df['cycle_duration_days']).fillna(0)
filtered_recs_df['overflow_flag'] = (filtered_recs_df['collection_fill_percentage'] >= 80).astype(int)

In [5]:
print("Shape:", filtered_recs_df.shape)

from IPython.display import display

display(filtered_recs_df.head())
display(filtered_recs_df.tail())

Shape: (1992, 7)


,ContainerID,cycle_duration_days,Avg_Dist,Spearman,collection_fill_percentage,avg_daily_fill_growth,overflow_flag
1,1000,21,79.150000,52.469086,100.0,4.761905,1
2,1000,4,76.655172,69.528848,44.0,11.000000,0
3,1000,7,79.076923,83.875464,72.0,10.285714,0
4,1000,8,75.903226,82.815132,100.0,12.500000,1
5,1000,6,87.666667,77.230257,100.0,16.666667,1


,ContainerID,cycle_duration_days,Avg_Dist,Spearman,collection_fill_percentage,avg_daily_fill_growth,overflow_flag
1988,10133,46,86.064935,87.843838,52.0,1.130435,0
1989,10133,48,70.642857,75.844559,62.5,1.302083,0
1990,10133,29,83.266667,70.043324,86.0,2.965517,1
1991,10133,15,91.937500,61.200847,70.0,4.666667,0
1992,10133,8,100.000000,100.000000,100.0,12.500000,1


In [6]:
# save as csv file
filtered_recs_df.to_csv("Cleaned-data/tableB.csv", index=False)